# Feature Engineering auf User-Level (Woche 2)

Nachdem wir die Daten bereinigt haben (`session_base.csv` und `not_canceled_trips.csv`), werden wir nun alle Events so aggregieren, dass jede Zeile genau einem Nutzer (`user_id`) entspricht. 

Wir werden neue Features erschaffen, die die Eigenschaften und das Verhalten des Kunden widerspiegeln.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Wir laden die gereinigten Daten aus Woche 1
sessions = pd.read_csv('../data/session_base.csv')
trips = pd.read_csv('../data/not_canceled_trips.csv')

print(f"Sessions: {len(sessions)}, Trips: {len(trips)}")

Sessions: 150000, Trips: 7614


## 1. Features aus allen Interaktionen (`session_base`)
Wir ermitteln, wie oft ein User die Seite besucht, wie oft er storniert (Cancellation Rate) und wie oft er nur "Window Shopping" betreibt (Sessions, bei denen er gar nichts bucht).

In [3]:
# Window-Shopping (Nichts gebucht)
sessions['is_window_shopping'] = ((sessions['flight_booked'] == False) & (sessions['hotel_booked'] == False)).astype(int)
sessions['has_discount'] = (sessions['flight_discount'] | sessions['hotel_discount']).astype(int)
sessions['cancellation'] = sessions['cancellation'].astype(int)

user_sessions = sessions.groupby('user_id').agg(
    total_sessions=('session_id', 'count'),
    total_cancellations=('cancellation', 'sum'),
    window_shopping_sessions=('is_window_shopping', 'sum'),
    total_discount_sessions=('has_discount', 'sum'),
    total_page_clicks=('page_clicks', 'sum'),
    avg_page_clicks=('page_clicks', 'mean')
).reset_index()

# Ratios (Raten)
user_sessions['cancellation_rate'] = user_sessions['total_cancellations'] / user_sessions['total_sessions']
user_sessions['window_shopping_rate'] = user_sessions['window_shopping_sessions'] / user_sessions['total_sessions']
user_sessions['discount_affinity'] = user_sessions['total_discount_sessions'] / user_sessions['total_sessions']

user_sessions.head()

,user_id,total_sessions,total_cancellations,window_shopping_sessions,total_page_clicks,avg_page_clicks,cancellation_rate,window_shopping_rate
0,12,1,0,1,3,3.000000,0.0,1.0
1,16,5,0,5,62,12.400000,0.0,1.0
2,19,1,0,0,12,12.000000,0.0,0.0
3,33,3,0,3,10,3.333333,0.0,1.0
4,37,5,0,5,29,5.800000,0.0,1.0


## 2. Features aus den tatsächlichen Reisen (`not_canceled_trips`)
Wir betrachten nun, ob der Kunde eher geschäftlich unterwegs ist (Fliegt alleine, wenig Gepäck, kurz) oder ein Familien-Trip macht (viele Sitze, mehrere Räume). Zudem berechnen wir den **Customer Lifetime Value (Total Spend)**.

In [ ]:
# Gesamtkosten
trips['flight_cost'] = trips['base_fare_usd'].fillna(0)
trips['hotel_cost'] = (trips['hotel_per_room_usd'] * trips['rooms'] * trips['nights']).fillna(0)
trips['trip_total_cost'] = trips['flight_cost'] + trips['hotel_cost']

# Business-Trip Annahme: Fliegt alleine (1 Seat) UND wenig/kein Aufgabegepäck (<=1) UND eher kurz (Nights <= 3)
trips['is_business_trip'] = (
    (trips['flight_booked'] == True) & 
    (trips['seats'] == 1) & 
    (trips['checked_bags'].fillna(0) <= 1) &
    (trips['nights'].fillna(0) <= 3)
).astype(int)

# Family-Trip Annahme: Mehr als 2 Sitze oder mehr als 1 Zimmer gebucht
trips['is_family_trip'] = (
    (trips['seats'] > 2) | 
    (trips['rooms'] > 1)
).astype(int)

# Aggregieren!
# Vorlaufzeit der Buchung berechnen (Booking Lead Time)
trips['session_start'] = pd.to_datetime(trips['session_start'])
trips['flight_start'] = pd.to_datetime(trips['departure_time'])
trips['hotel_start'] = pd.to_datetime(trips['check_in_time'])
trips['travel_start'] = trips[['flight_start', 'hotel_start']].min(axis=1)
trips['booking_lead_time'] = (trips['travel_start'] - trips['session_start']).dt.days
trips['booking_lead_time'] = trips['booking_lead_time'].fillna(0)

user_trips = trips.groupby('user_id').agg(
    total_trips=('trip_id', 'count'),
    total_spent=('trip_total_cost', 'sum'),
    avg_spent_per_trip=('trip_total_cost', 'mean'),
    avg_nights_hotel=('nights', 'mean'),
    avg_booking_lead_time=('booking_lead_time', 'mean'),
    total_business_trips=('is_business_trip', 'sum'),
    total_family_trips=('is_family_trip', 'sum')
).reset_index()

user_trips.head()

,user_id,total_trips,total_spent,avg_spent_per_trip,avg_nights_hotel,total_business_trips,total_family_trips
0,19,1,2650.00,2650.00,10.0,0,0
1,44,1,253.60,253.60,4.0,0,0
2,287,1,637.31,637.31,2.0,1,0
3,321,1,1631.00,1631.00,7.0,0,0
4,324,1,1965.65,1965.65,1.0,0,1


: 

## 3. Beide Tabellen zusammenfügen (Left Join)
Wir führen einen **Left Join** durch, da wir alle Besucher (aus `session_base.csv`) behalten wollen, auch diejenigen, die noch gar keine Reise gebucht haben (`total_trips` = 0). 

In [ ]:
user_base = pd.merge(user_sessions, user_trips, on='user_id', how='left')

# Auffüllen der fehlenden Werte für Leute, die nichts gekauft haben
fill_cols = ['total_trips', 'total_spent', 'avg_spent_per_trip', 'avg_nights_hotel', 'avg_booking_lead_time', 'total_business_trips', 'total_family_trips']
user_base[fill_cols] = user_base[fill_cols].fillna(0)

# Verhältnisse (Ratios) berechnen
user_base['business_trip_ratio'] = np.where(user_base['total_trips'] > 0, user_base['total_business_trips'] / user_base['total_trips'], 0)
user_base['family_trip_ratio'] = np.where(user_base['total_trips'] > 0, user_base['total_family_trips'] / user_base['total_trips'], 0)

print(f"Finale User Base Shape: {user_base.shape}")
display(user_base.head())

## 4. Feature Visualisierung
Wie verhalten sich unsere generierten Features in der Praxis? Gibt es logische Zusammenhänge?

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(user_base[user_base['total_spent'] > 0]['total_spent'], bins=50, kde=True)
plt.title('Logische Verteilung des Total-Spends')
plt.xlabel('Geld ($)')
plt.xlim(0, 10000)
plt.show()

Das sieht gut aus. Wir können die CSV-Datei nun abspeichern!

In [ ]:
output_path = '../data/user_base.csv'
user_base.to_csv(output_path, index=False)
print("✅ User-Dataset erfolgreich als 'user_base.csv' exportiert.")